In [71]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

# -----------------------------
# 🌐 1. CONFIGURACIÓN DE URL Y HEADERS
# -----------------------------
url = "https://es.wikipedia.org/wiki/Anexo:Torneo_Apertura_2026_(Colombia)_-_Fase_todos_contra_todos"
headers = {"User-Agent": "Mozilla/5.0"}

# -----------------------------
# 📡 2. PETICIÓN HTTP A WIKIPEDIA
# -----------------------------
response = requests.get(url, headers=headers)

if response.status_code != 200:
    raise Exception(f"Error en la request: {response.status_code}")

# -----------------------------
# 🥣 3. PARSEO DE HTML CON BEAUTIFULSOUP
# -----------------------------
soup = BeautifulSoup(response.text, "lxml")

filas = soup.find_all("tr")
partidos = []

# -----------------------------
# 🔍 4. DEFINICIÓN DE REGEX PARA EXTRACCIÓN
# -----------------------------
# Este patrón captura: Fecha, Local, Marcador (Goles) y el resto de la info (Visitante/Estadio)
patron = r"(\d{1,2} de \w+)\s+(.*?)\s+(\d+[:\-–]\d+)(?:\s+\(.*?\))?\s+(.*)"

# -----------------------------
# 🔄 5. ITERACIÓN Y PROCESAMIENTO DE FILAS
# -----------------------------
for fila in filas:
    texto = fila.get_text(" ", strip=True)
    match = re.search(patron, texto)

    if match:
        try:
            # -----------------------------
            # ✂️ 6. SEGMENTACIÓN DE GRUPOS CAPTURADOS
            # -----------------------------
            fecha = match.group(1)
            local = match.group(2)
            marcador = match.group(3)
            resto = match.group(4)

            # -----------------------------
            # ⚽ 7. LIMPIEZA DE GOLES Y MARCADOR
            # -----------------------------
            goles_local, goles_visitante = re.split("[:-–]", marcador)

            # -----------------------------
            # 🏟️ 8. EXTRACCIÓN DE VISITANTE Y ESTADIO
            # -----------------------------
            # Separamos el nombre del equipo visitante de la información del estadio
            visitante = re.split(r"Estadio|Coliseo", resto)[0].strip()
            estadio = resto.replace(visitante, "").strip()

            visitante = visitante.split(",")[0].strip()
            estadio = estadio.strip()

            # -----------------------------
            # 📝 9. CONSTRUCCIÓN DE LA LISTA DE DICCIONARIOS
            # -----------------------------
            partidos.append({
                "fecha": fecha,
                "local": local.strip(),
                "visitante": visitante.strip(),
                "estadio": estadio,
                "goles_local": int(goles_local),
                "goles_visitante": int(goles_visitante)
            })

        except:
            continue

# -----------------------------
# 📊 10. CREACIÓN Y LIMPIEZA DEL DATAFRAME
# -----------------------------
# Convertimos a DataFrame y eliminamos filas repetidas para tener el df_final limpio
df = pd.DataFrame(partidos).drop_duplicates()

In [72]:
# -----------------------------
# 🏗️ 11. DESCOMPOSICIÓN DE UBICACIÓN (ESTADIO Y CIUDAD)
# -----------------------------
# Separamos la columna 'estadio' en dos nuevas columnas usando la coma como delimitador.
# n=1 asegura que solo divida en la primera coma que encuentre.
df[["estadio_nombre", "ciudad"]] = df["estadio"].str.split(",", n=1, expand=True)

# -----------------------------
# 🧹 12. NORMALIZACIÓN Y LIMPIEZA DE TEXTO
# -----------------------------
# Quitamos la palabra "Estadio" para dejar solo el nombre propio (ej: "El Campín") 
# y aplicamos .strip() para eliminar espacios en blanco sobrantes en ambas columnas.
df["estadio_nombre"] = df["estadio_nombre"].str.replace("Estadio", "").str.strip()
df["ciudad"] = df["ciudad"].str.strip()

In [73]:
display(df)

,fecha,local,visitante,estadio,goles_local,goles_visitante,estadio_nombre,ciudad
0,16 de enero,Deportivo Pereira,Llaneros,"Estadio Centenario , Armenia",0,2,Centenario,Armenia
1,16 de enero,Fortaleza,Alianza Valledupar,"Estadio Metropolitano de Techo , Bogotá",1,0,Metropolitano de Techo,Bogotá
2,17 de enero,América de Cali,Internacional de Bogotá,"Estadio Pascual Guerrero , Cali",3,0,Pascual Guerrero,Cali
3,17 de enero,Atlético Nacional,Boyacá Chicó,"Estadio Atanasio Girardot , Medellín",4,0,Atanasio Girardot,Medellín
4,17 de enero,Atlético Bucaramanga,Millonarios,"Estadio Américo Montanini , Bucaramanga",2,1,Américo Montanini,Bucaramanga
...,...,...,...,...,...,...,...,...
145,30 de marzo,Independiente Medellín,América de Cali,"Estadio Atanasio Girardot , Medellín",2,1,Atanasio Girardot,Medellín
146,30 de marzo,Millonarios,Fortaleza,"Estadio El Campín , Bogotá",2,2,El Campín,Bogotá
147,31 de marzo,Deportes Tolima,Águilas Doradas,"Estadio Manuel Murillo Toro , Ibagué",4,1,Manuel Murillo Toro,Ibagué
148,1 de marzo,Deportivo Cali,Fortaleza,"Estadio Deportivo Cali , Palmira",1,1,Deportivo Cali,Palmira


In [74]:
# -----------------------------
# 📅 13. DICCIONARIO DE TRADUCCIÓN DE MESES
# -----------------------------
# Definimos un mapeo manual para convertir los nombres de los meses en español 
# a su representación numérica ISO (01 a 12).
meses = {
    "enero": "01", "febrero": "02", "marzo": "03",
    "abril": "04", "mayo": "05", "junio": "06",
    "julio": "07", "agosto": "08", "septiembre": "09",
    "octubre": "10", "noviembre": "11", "diciembre": "12"
}

# -----------------------------
# ✂️ 14. EXTRACCIÓN DE COMPONENTES DE FECHA
# -----------------------------
# Utilizamos expresiones regulares (.extract) para separar el número del día 
# y el nombre del mes que vienen en el texto capturado de Wikipedia.
df[["dia", "mes_txt"]] = df["fecha"].str.extract(r"(\d{1,2}) de (\w+)")

# -----------------------------
# 🗺️ 15. MAPEADO DE MESES A FORMATO NUMÉRICO
# -----------------------------
# Normalizamos el texto del mes a minúsculas y aplicamos el diccionario 
# para obtener el número correspondiente (ej: "marzo" -> "03").
df["mes"] = df["mes_txt"].str.lower().map(meses)

# -----------------------------
# 🗓️ 16. CONSTRUCCIÓN DE OBJETO DATETIME
# -----------------------------
# Concatenamos el año (2026), el mes y el día (usando .zfill(2) para asegurar el 
# formato de dos dígitos) para crear una columna de fecha real reconocida por Pandas.
df["fecha_dt"] = pd.to_datetime(
    "2026-" + df["mes"] + "-" + df["dia"].str.zfill(2),
    errors="coerce"
)

# -----------------------------
# 🧹 17. LIMPIEZA DE COLUMNAS AUXILIARES
# -----------------------------
# Eliminamos las columnas temporales que creamos para el proceso de extracción, 
# dejando el DataFrame final optimizado y solo con la columna 'fecha_dt' lista.
df.drop(columns=["dia", "mes_txt", "mes"], inplace=True)

# -----------------------------
# 🔍 18. VERIFICACIÓN DE RESULTADOS
# -----------------------------
# Imprimimos las primeras filas para confirmar que la conversión de texto a 
# datetime se realizó correctamente.
print(df[["fecha", "fecha_dt"]].head())

         fecha   fecha_dt
0  16 de enero 2026-01-16
1  16 de enero 2026-01-16
2  17 de enero 2026-01-17
3  17 de enero 2026-01-17
4  17 de enero 2026-01-17


In [75]:
# -----------------------------
# ⚽ 19. CÁLCULO DE DIFERENCIA DE GOLES
# -----------------------------
# Creamos una columna numérica que resta los goles del visitante a los del local.
# Si es positivo (>0) ganó el local, si es 0 hubo empate, si es negativo (<0) ganó la visita.
df["resultado"] = df["goles_local"] - df["goles_visitante"]

# -----------------------------
# 🏷️ 20. ETIQUETADO CATEGÓRICO DEL RESULTADO
# -----------------------------
# Transformamos la diferencia numérica en una etiqueta de texto legible.
# Esto facilita el cálculo de precisión (Accuracy) y la visualización en Streamlit.
df["resultado_label"] = df["resultado"].apply(
    lambda x: "Local" if x > 0 else ("Empate" if x == 0 else "Visitante")
)

In [76]:
# -----------------------------
# 🏗️ 21. SELECCIÓN DE VARIABLES CRÍTICAS (FEATURE SELECTION)
# -----------------------------
# Creamos el DataFrame definitivo ('df_final') filtrando únicamente las columnas 
# que aportan valor real al modelo predictivo y a la interfaz de Streamlit.
# Con esto eliminamos ruidos, columnas temporales y datos crudos (raw) ya procesados.
df_final = df[
    [
        "fecha",           # Referencia temporal para el usuario
        "local",           # Identificador equipo A
        "visitante",       # Identificador equipo B
        "goles_local",     # Variable para calcular Ataque/Defensa
        "goles_visitante", # Variable para calcular Ataque/Defensa
        "estadio_nombre",  # Contexto geográfico
        "ciudad",          # Variable para futuros ajustes (clima/altura)
        "resultado_label"  # Variable objetivo (Target) para validación
    ]
]

In [77]:
display(df_final)

,fecha,local,visitante,goles_local,goles_visitante,estadio_nombre,ciudad,resultado_label
0,16 de enero,Deportivo Pereira,Llaneros,0,2,Centenario,Armenia,Visitante
1,16 de enero,Fortaleza,Alianza Valledupar,1,0,Metropolitano de Techo,Bogotá,Local
2,17 de enero,América de Cali,Internacional de Bogotá,3,0,Pascual Guerrero,Cali,Local
3,17 de enero,Atlético Nacional,Boyacá Chicó,4,0,Atanasio Girardot,Medellín,Local
4,17 de enero,Atlético Bucaramanga,Millonarios,2,1,Américo Montanini,Bucaramanga,Local
...,...,...,...,...,...,...,...,...
145,30 de marzo,Independiente Medellín,América de Cali,2,1,Atanasio Girardot,Medellín,Local
146,30 de marzo,Millonarios,Fortaleza,2,2,El Campín,Bogotá,Empate
147,31 de marzo,Deportes Tolima,Águilas Doradas,4,1,Manuel Murillo Toro,Ibagué,Local
148,1 de marzo,Deportivo Cali,Fortaleza,1,1,Deportivo Cali,Palmira,Empate


In [78]:
df_final['local'].unique()

array(['Deportivo Pereira', 'Fortaleza', 'América de Cali',
       'Atlético Nacional', 'Atlético Bucaramanga', 'Deportivo Pasto',
       'Jaguares', 'Junior', 'Santa Fe', 'Cúcuta Deportivo',
       'Internacional de Bogotá', 'Deportes Tolima', 'Llaneros',
       'Deportivo Cali', 'Boyacá Chicó', 'Águilas Doradas', 'Millonarios',
       'Once Caldas', 'a las', 'Independiente Medellín',
       'Alianza Valledupar',
       'por lluvia torrencial. Se reanudó el 2 de febrero a las',
       'Indpeendiente Medellín',
       'por tormenta eléctrica. Se reanudó el 30 de marzo a las'],
      dtype=object)

In [80]:
import pandas as pd
import re

# -----------------------------
# 🛡️ 22. CREACIÓN DE COPIA DE SEGURIDAD
# -----------------------------
# Trabajamos sobre una copia ('df_clean') para no alterar el DataFrame original 
# hasta estar seguros de que la limpieza es correcta.
df_clean = df_final.copy()

# -----------------------------
# 🧹 23. NORMALIZACIÓN DE TEXTO BÁSICA
# -----------------------------
# Eliminamos espacios en blanco accidentales al inicio o al final de los nombres.
df_clean["local"] = df_clean["local"].str.strip()
df_clean["visitante"] = df_clean["visitante"].str.strip()

# -----------------------------
# 🚫 24. FILTRADO DE RUIDO Y ESTADOS DE PARTIDO
# -----------------------------
# Wikipedia a veces incluye notas como "suspendido por lluvia". 
# Este regex detecta y elimina filas que contienen estas palabras clave en lugar de equipos.
patron_basura = r"a las|lluvia|tormenta|reanudó|suspendido"

df_clean = df_clean[
    ~df_clean["local"].str.contains(patron_basura, case=False, na=False) &
    ~df_clean["visitante"].str.contains(patron_basura, case=False, na=False)
]

# -----------------------------
# 🔧 25. CORRECCIÓN MANUAL DE TYPOS (ERRATAS)
# -----------------------------
# Definimos un diccionario para unificar nombres mal escritos. 
# Esto es vital para que el historial de puntos se asigne al equipo correcto.
correcciones = {
    "Indpeendiente Medellín": "Independiente Medellín",
    "Independiente  Medellín": "Independiente Medellín",
}

df_clean["local"] = df_clean["local"].replace(correcciones)
df_clean["visitante"] = df_clean["visitante"].replace(correcciones)

# -----------------------------
# 🧠 26. FUNCIÓN DE LIMPIEZA DE SUFIJOS
# -----------------------------
# Definimos una lógica para cortar cualquier texto sobrante que el scraper 
# haya capturado después del nombre del equipo (como menciones a estadios).
def limpiar_equipo(texto):
    # Cortamos el texto en la primera aparición de 'Estadio', 'Coliseo' o comas.
    texto = re.split(r"Estadio|Coliseo|\,", texto)[0]
    return texto.strip()

df_clean["local"] = df_clean["local"].apply(limpiar_equipo)
df_clean["visitante"] = df_clean["visitante"].apply(limpiar_equipo)

# -----------------------------
# 🔍 27. FILTRO DE INTEGRIDAD POR LONGITUD
# -----------------------------
# Eliminamos registros donde el nombre del equipo sea demasiado corto (ruido), 
# asegurando que solo queden nombres de equipos reales.
df_clean = df_clean[
    (df_clean["local"].str.len() > 3) &
    (df_clean["visitante"].str.len() > 3)
]

# -----------------------------
# 🧪 28. VALIDACIÓN DE CONSISTENCIA (SET DE EQUIPOS)
# -----------------------------
# Creamos un conjunto único de todos los equipos para revisar visualmente 
# que no existan duplicados o nombres extraños.
equipos = sorted(set(df_clean["local"]) | set(df_clean["visitante"]))

print("Equipos detectados:\n")
for e in equipos:
    print(e)

# -----------------------------
# ✨ 29. ACTUALIZACIÓN DEL DATAFRAME FINAL
# -----------------------------
# Una vez limpio, sobreescribimos 'df_final' con la versión procesada. 
# ¡Ya tienes datos de alta fidelidad para el Poisson!
df_final = df_clean.copy()

print("\n✅ DataFrame limpio listo:")
#print(df_final.head())

Equipos detectados:

Alianza Valledupar
América de Cali
Atlético Bucaramanga
Atlético Nacional
Boyacá Chicó
Cúcuta Deportivo
Deportes Tolima
Deportivo Cali
Deportivo Pasto
Deportivo Pereira
Fortaleza
Independiente Medellín
Internacional de Bogotá
Jaguares
Junior
Llaneros
Millonarios
Once Caldas
Santa Fe
Águilas Doradas

✅ DataFrame limpio listo:


In [82]:
import pandas as pd

# -----------------------------
# 🛡️ 30. COPIA DE SEGURIDAD PARA LA TABLA
# -----------------------------
# Creamos 'df_tabla' para procesar las estadísticas sin afectar el histórico de partidos.
df_tabla = df_final.copy()

# -----------------------------
# 📅 31. ORDENAMIENTO CRONOLÓGICO
# -----------------------------
# Ordenamos por fecha para que el cálculo de la "forma" (últimos 5 partidos) sea correcto.
df_tabla = df_tabla.sort_values("fecha")

# -----------------------------
# ⚽ 32. GENERACIÓN DE VARIABLES BASE (PG, PE, PP)
# -----------------------------
# Calculamos Partidos Jugados, victorias local/visitante, empates y derrotas.
# Usamos .astype(int) para convertir los booleanos (True/False) en números (1/0).
df_tabla["PJ"] = 1
df_tabla["PG_local"] = (df_tabla["goles_local"] > df_tabla["goles_visitante"]).astype(int)
df_tabla["PG_visitante"] = (df_tabla["goles_visitante"] > df_tabla["goles_local"]).astype(int)
df_tabla["PE"] = (df_tabla["goles_local"] == df_tabla["goles_visitante"]).astype(int)
df_tabla["PP_local"] = (df_tabla["goles_local"] < df_tabla["goles_visitante"]).astype(int)
df_tabla["PP_visitante"] = (df_tabla["goles_visitante"] < df_tabla["goles_local"]).astype(int)

# Asignación de puntos (3 por ganar, 1 por empatar) y conteo de goles.
df_tabla["PTS_local"] = df_tabla["PG_local"] * 3 + df_tabla["PE"]
df_tabla["PTS_visitante"] = df_tabla["PG_visitante"] * 3 + df_tabla["PE"]
df_tabla["GF_local"] = df_tabla["goles_local"]
df_tabla["GC_local"] = df_tabla["goles_visitante"]
df_tabla["GF_visitante"] = df_tabla["goles_visitante"]
df_tabla["GC_visitante"] = df_tabla["goles_local"]

# -----------------------------
# 🔄 33. UNIFICACIÓN DE DATOS (LOCAL + VISITANTE)
# -----------------------------
# Separamos el rendimiento de local y de visita para luego unificarlos en una sola lista de 'equipos'.
local = df_tabla[[
    "fecha", "local", "PJ", "PG_local", "PE", "PP_local",
    "GF_local", "GC_local", "PTS_local"
]].rename(columns={
    "local": "equipo", "PG_local": "PG", "PP_local": "PP",
    "GF_local": "GF", "GC_local": "GC", "PTS_local": "PTS"
})

visitante = df_tabla[[
    "fecha", "visitante", "PJ", "PG_visitante", "PE", "PP_visitante",
    "GF_visitante", "GC_visitante", "PTS_visitante"
]].rename(columns={
    "visitante": "equipo", "PG_visitante": "PG", "PP_visitante": "PP",
    "GF_visitante": "GF", "GC_visitante": "GC", "PTS_visitante": "PTS"
})

# Concatenamos ambos para tener el rendimiento total por equipo.
df_equipos = pd.concat([local, visitante])

# -----------------------------
# 🏆 34. AGREGACIÓN Y CÁLCULO DE TABLA GENERAL
# -----------------------------
# Agrupamos por equipo y sumamos todas las estadísticas acumuladas.
tabla = df_equipos.groupby("equipo").agg({
    "PJ": "sum", "PG": "sum", "PE": "sum", "PP": "sum",
    "GF": "sum", "GC": "sum", "PTS": "sum"
})

# Calculamos la Diferencia de Goles (DG).
tabla["DG"] = tabla["GF"] - tabla["GC"]

# -----------------------------
# 🔢 35. ORDENAMIENTO OFICIAL Y POSICIONES
# -----------------------------
# Aplicamos los criterios de desempate: 1. Puntos, 2. DG, 3. Goles a Favor.
tabla = tabla.sort_values(["PTS", "DG", "GF"], ascending=[False, False, False])
tabla["Pos"] = range(1, len(tabla) + 1)
tabla = tabla.reset_index()

# -----------------------------
# 🔥 36. CÁLCULO DE LA FORMA (ÚLTIMOS 5 PARTIDOS)
# -----------------------------
# Determinamos si cada partido fue Ganado (G), Empatado (E) o Perdido (P).
df_equipos["resultado"] = df_equipos.apply(
    lambda x: "G" if x["GF"] > x["GC"] else ("E" if x["GF"] == x["GC"] else "P"), axis=1
)

# Concatenamos los resultados de los últimos 5 encuentros en una cadena (ej: "GGGPE").
forma = (
    df_equipos.sort_values("fecha")
    .groupby("equipo")["resultado"]
    .apply(lambda x: "".join(x.tail(5)))
    .reset_index()
)

# Unimos la columna de forma a nuestra tabla principal.
tabla = tabla.merge(forma, on="equipo", how="left")

# -----------------------------
# 🎯 37. COLUMNAS FINALES Y RESULTADO
# -----------------------------
# Reorganizamos las columnas para una visualización limpia y profesional.
tabla = tabla[[
    "Pos", "equipo", "PJ", "PG", "PE", "PP",
    "GF", "GC", "DG", "PTS", "resultado"
]]

print(tabla)

    Pos                   equipo  PJ  PG  PE  PP  GF  GC  DG  PTS resultado
0     1        Atlético Nacional  13  10   0   3  28  10  18   30     GGGGG
1     2          Deportivo Pasto  14   8   3   3  21  17   4   27     GGPEG
2     3          Deportes Tolima  14   7   5   2  22  11  11   26     EEGGP
3     4                   Junior  14   8   1   5  20  18   2   25     GGGEG
4     5              Once Caldas  14   6   6   2  25  18   7   24     GEEGE
5     6              Millonarios  14   6   3   5  25  16   9   21     GPEEG
6     7          América de Cali  13   6   3   4  17  10   7   21     GEPPP
7     8  Internacional de Bogotá  14   5   6   3  17  19  -2   21     GGPEG
8     9     Atlético Bucaramanga  13   4   7   2  17  10   7   19     PEPGE
9    10           Deportivo Cali  14   5   4   5  15  12   3   19     GGPPE
10   11          Águilas Doradas  14   5   4   5  14  16  -2   19     GGPGP
11   12                 Santa Fe  13   4   6   3  16  16   0   18     PGEGE
12   13     

In [83]:
#pip install scipy